# `google.colab.ai` pre-flight

*Linguistic Data Analysis II — pre-course capacity check (run once, on the Tohoku Google account students will use).*

**Purpose.** Before building more on top of Colab's built-in Gemini, confirm the backend the day-notebooks call — `from google.colab import ai; ai.generate_text(p)` — is reachable on the Tohoku account, find where (if anywhere) it throttles, and measure how stable the parsed CEFR labels are run-to-run.

Run **Runtime → Run all**. Each step prints a compact report; the final cell summarizes go / no-go.

> This notebook only runs **inside Google Colab** — `google.colab.ai` does not exist elsewhere. If Step 2 fails, the whole `colab.ai` path is unavailable for Tohoku accounts → fall back to the Gemini-API key (D4) and document it.

---

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Reuses the SAME label parser as the day-notebooks, so the numbers below
# reflect what the course actually runs.
import json, re, time, inspect, urllib.request, statistics, random
from collections import Counter
from google.colab import ai   # Colab's built-in Gemini, no API key

# --- What can we actually tune on THIS runtime? Print the real signature.
#     (Don't trust blog posts — ask the installed library.)
sig = inspect.signature(ai.generate_text)
print("ai.generate_text signature:", sig)
params = sig.parameters

# Change ONE knob at a time (ablation). We start with temperature only and
# leave the DEFAULT model + seed OUT, because forcing model_name / seed all
# at once previously *increased* variance and made the cause unattributable.
#   temperature=0 → greedy decoding (removes sampling randomness)
# Even so, output on a hosted, batched endpoint is only BEST-EFFORT
# reproducible — never bit-guaranteed (requests are batched with others and
# float reductions reorder).
USE_TEMPERATURE = True   #@param {type:"boolean"}
USE_SEED        = False  #@param {type:"boolean"}
USE_MODEL_NAME  = False  #@param {type:"boolean"}
MODEL_NAME      = "google/gemini-2.5-flash"  #@param {type:"string"}

GEN_KWARGS = {}
if USE_TEMPERATURE and "temperature" in params: GEN_KWARGS["temperature"] = 0
if USE_SEED        and "seed" in params:        GEN_KWARGS["seed"] = 42
if USE_MODEL_NAME  and "model_name" in params:  GEN_KWARGS["model_name"] = MODEL_NAME
print("Passing to generate_text:", GEN_KWARGS or "(prompt only — default model)")
for k in ("temperature", "seed", "model_name"):
    if k not in params:
        print(f"  note: this runtime\u2019s generate_text has no '{k}' parameter.")

def generate_text(p):
    return ai.generate_text(p, **GEN_KWARGS)

def _extract_level(text):
    """Identical to the day-notebooks' parser."""
    m = re.search(r"\b([ABC][12])\b", str(text).upper())
    return m.group(1) if m else "??"

LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
PROMPT = (
    "You are a CEFR rater. Reply with ONLY one label from "
    "A1, A2, B1, B2, C1, C2 for this sentence.\n\nSentence: {text}\nLabel:"
)

# Real CEFR sentences from the course gold set (cycled to reach the call counts).
GOLD_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/cefr_sentences.json"
gold = json.loads(urllib.request.urlopen(GOLD_URL).read().decode("utf-8"))
print(f"\nLoaded {len(gold)} gold sentences. Backend: Colab Gemini (colab.ai).")
print("temperature=0 set — output should be near-reproducible (best-effort)."
      if "temperature" in GEN_KWARGS else
      "No temperature control on this version → non-deterministic by design.")

# ------------------------------------------------------------------ resilience
# The same guards the day-notebooks and the mini-project template use, so what we
# measure below is what students will actually experience. Explained in full in
# resources/extra/handling-rate-limits.ipynb.

def looks_like_rate_limit(error):
    text = str(error).lower()
    return any(s in text for s in
               ["429", "resource_exhausted", "rate limit", "quota", "too many requests"])

def looks_like_daily_quota(error):
    text = str(error).lower()
    return "perday" in text.replace(" ", "") or "per day" in text

def suggested_delay(error, fallback):
    m = re.search(r"retry in ([0-9]+(?:\.[0-9]+)?)s", str(error).lower())
    return float(m.group(1)) if m else fallback

def with_pacing(call, min_interval):
    """At least min_interval seconds between consecutive calls."""
    last = [0.0]
    def paced(prompt):
        wait = min_interval - (time.monotonic() - last[0])
        if wait > 0:
            time.sleep(wait)
        last[0] = time.monotonic()
        return call(prompt)
    return paced

def with_retry(call, max_retries=5, base_delay=4.0, verbose=True):
    """Back off and retry per-minute limits; fail fast on daily caps and real bugs."""
    def retrying(prompt):
        for attempt in range(max_retries + 1):
            try:
                return call(prompt)
            except Exception as error:
                if not looks_like_rate_limit(error):
                    raise                       # a real bug — surface it now
                if looks_like_daily_quota(error):
                    raise RuntimeError(
                        "DAILY quota exhausted — waiting cannot help today.") from error
                if attempt == max_retries:
                    raise
                delay = suggested_delay(error, base_delay * (2 ** attempt))
                delay += random.uniform(0, 0.3 * delay)      # jitter
                if verbose:
                    print(f"    (rate limited — waiting {delay:.1f}s)")
                time.sleep(delay)
        raise RuntimeError("unreachable")
    return retrying

# colab.ai publishes no RPM number, so pace conservatively at the free-tier
# gemini-2.5-flash rate (5 RPM -> 12s) and let Step 3 tell us if that is wrong.
ASSUMED_RPM   = 5      #@param {type:"integer"}
MIN_INTERVAL  = 60.0 / ASSUMED_RPM

generate_raw = generate_text          # unguarded — Step 3 needs this to find the ceiling
generate_guarded = with_pacing(with_retry(generate_raw), MIN_INTERVAL)
print(f"\nGuarded backend ready: pacing at {MIN_INTERVAL:.1f}s/call "
      f"(assuming {ASSUMED_RPM} RPM), retry+backoff on 429.")

## Step 2 — Smoke test (is the API even reachable?)

One list-models call + one generate call. If this errors, **stop** — the `colab.ai` path is unavailable on this account.

In [ ]:
#@title Step 2 — smoke test
try:
    print("Available models:")
    try:
        print(" ", ai.list_models())
    except Exception as e:
        print("  (list_models not available:", type(e).__name__, e, ")")
    print("\nSingle call →", repr(generate_text("Say OK")))
    print("\n✅ Reachable.")
except Exception as e:
    print("❌ colab.ai unavailable on this account:", type(e).__name__, e)
    print("   → Escalate to the Gemini-API .env fallback (D4) and stop.")

## Step 3 — Quota ceiling (100-call loop, deliberately unguarded)

Fire 100 real CEFR prompts as fast as they will go, with **no pacing and no
retry**. That is the point: this step exists to *find* the ceiling, so guarding it
would destroy the measurement.

Each failure is caught and classified — per-minute vs per-day — because the two
mean very different things for the course. A per-minute ceiling is a pacing
problem students can work around; a per-day ceiling is a hard budget that caps how
much lab work is possible per student per day.

Step 3b then re-runs the same load *with* the guards, to confirm the workaround
actually holds on this account.

In [ ]:
#@title Step 3 — unguarded 100-call quota + latency probe
N = 100
items = [gold[i % len(gold)] for i in range(N)]

latencies, preds, first_fail = [], [], None
fails = {"per_minute": 0, "per_day": 0, "other": 0}
t0 = time.time()
for i, item in enumerate(items, 1):
    prompt = PROMPT.format(text=item["text"])
    c0 = time.time()
    try:
        reply = generate_raw(prompt)
        latencies.append(time.time() - c0)
        preds.append(_extract_level(reply))
    except Exception as e:
        kind = ("per_day" if looks_like_daily_quota(e)
                else "per_minute" if looks_like_rate_limit(e) else "other")
        fails[kind] += 1
        if first_fail is None:
            first_fail = (i, kind, type(e).__name__, str(e)[:160])
        preds.append("ERR")
    if i % 20 == 0:
        print(f"  ...{i}/{N}  ({sum(fails.values())} failures so far)")
wall = time.time() - t0

ok = len(latencies)
print(f"\n{ok}/{N} succeeded. Failures: {fails}")
if latencies:
    lat = sorted(latencies)
    p95 = lat[min(len(lat)-1, int(0.95*len(lat)))]
    print(f"latency: median {statistics.median(lat):.2f}s  p95 {p95:.2f}s  "
          f"min {lat[0]:.2f}s  max {lat[-1]:.2f}s")
print(f"total wall-time for {N} calls: {wall:.1f}s ({wall/N:.2f}s/call incl. overhead)")

if first_fail:
    i, kind, name, msg = first_fail
    print(f"first failure at call {i}: [{kind}] {name}: {msg}")
    if kind == "per_minute":
        observed_rpm = i - 1
        print(f"  -> throttled after ~{observed_rpm} calls in "
              f"{wall:.0f}s. Set ASSUMED_RPM at or below this and re-run Setup.")
    elif kind == "per_day":
        print("  -> DAILY cap. This is a hard budget: no pacing can widen it.")
        print("     Record the number and size the labs to fit it.")
else:
    print(f"first failure: none — no throttling in {N} unguarded calls")
print("parse rate (a valid CEFR label):", f"{sum(p in LEVELS for p in preds)}/{N}")

## Step 3b — Does pacing + backoff actually fix it?

Only meaningful if Step 3 hit a **per-minute** limit. Same 100 prompts, now
through `generate_guarded`. What we want is 100/100 — that would confirm the
guidance we give students (pace, back off, retry) genuinely works on this account
rather than merely sounding sensible.

This is slow by construction: at the paced rate it takes roughly
`100 x MIN_INTERVAL` seconds. That wall-time *is* the finding — it tells you how
long a full-gold-set lab run will really take.

Skip it if Step 3 hit a per-day cap; there is nothing left to spend today.

In [ ]:
#@title Step 3b — the same load, guarded
RUN_STEP_3B = True   #@param {type:"boolean"}
N_GUARDED   = 100    #@param {type:"integer"}

if not RUN_STEP_3B:
    print("skipped (RUN_STEP_3B = False)")
elif first_fail and first_fail[1] == "per_day":
    print("skipped — Step 3 hit a DAILY cap; no quota left to test with today.")
else:
    g_items = [gold[i % len(gold)] for i in range(N_GUARDED)]
    g_ok, g_fail, g_first_fail = 0, 0, None
    t0 = time.time()
    for i, item in enumerate(g_items, 1):
        try:
            _extract_level(generate_guarded(PROMPT.format(text=item["text"])))
            g_ok += 1
        except Exception as e:
            g_fail += 1
            if g_first_fail is None:
                g_first_fail = (i, type(e).__name__, str(e)[:160])
        if i % 20 == 0:
            print(f"  ...{i}/{N_GUARDED}  ({g_fail} failures, "
                  f"{time.time()-t0:.0f}s elapsed)")
    g_wall = time.time() - t0

    print(f"\nGuarded: {g_ok}/{N_GUARDED} succeeded, {g_fail} failed, "
          f"{g_wall:.0f}s ({g_wall/N_GUARDED:.1f}s/call).")
    if g_ok == N_GUARDED:
        print("✅ Pacing + backoff fully absorbed the limit. Ship this wrapper in the")
        print("   day-notebooks and the observed cost is "
              f"~{g_wall/60:.1f} min per 100 calls.")
    else:
        print(f"⚠️  Still failing ({g_first_fail}). MIN_INTERVAL is too aggressive —")
        print("   lower ASSUMED_RPM, re-run Setup, and try again.")

## Step 4 — Extrapolate to the class (8 students)

The single-user numbers above → what a class of 8 implies. **Caveat:** this measures one account sequentially; it cannot test 8-way *concurrency*. If `colab.ai` quota is per-account (likely), 8 students = 8 independent quotas and Step 3 passing is reassuring. If it's a shared project quota, real load is ~8× and only a multi-account trial can confirm it.

In [ ]:
#@title Step 4 — class-of-15 projection
CLASS = 8      # D3 (decided 2026-07-19): 8 students
CALLS_PER_STUDENT_PER_DAY = 72 + 72 + 60  # tutorial + few-shot + lab, rough
if latencies:
    per_call = wall / N
    student_day = per_call * CALLS_PER_STUDENT_PER_DAY
    print(f"~{per_call:.2f}s/call → one student\u2019s ~{CALLS_PER_STUDENT_PER_DAY} "
          f"calls/day ≈ {student_day/60:.1f} min of model time.")
    print(f"Class total: {CLASS*CALLS_PER_STUDENT_PER_DAY} calls/day across {CLASS} accounts.")
    print(f"If quota is PER-ACCOUNT: each student runs the {student_day/60:.1f}-min "
          "budget independently → fine.")
    print("If quota is SHARED/project: treat 100-call ceiling from Step 3 as the "
          "whole-class ceiling → likely too low, plan the .env fallback.")
else:
    print("No successful calls in Step 3 — cannot project. Fix Step 2/3 first.")

## Step 5 — Determinism (label stability)

`colab.ai` exposes no temperature knob, so raw text will vary. What matters for the course is whether the **parsed CEFR label** is stable. Send the same prompt 10× for a few sentences and report the modal label and its agreement rate — this is how much run-to-run F1 wobble to warn students about.

This step runs through `generate_guarded`, so a throttle shows up as a pause rather than as an `ERR` that would silently depress the agreement figure. We want to measure the model's *variability*, not our own rate-limit noise.

In [ ]:
#@title Step 5 — same prompt × 10, label stability (guarded)
REPEATS = 10
SAMPLES = 4
sample_items = [gold[i * (len(gold)//SAMPLES)] for i in range(SAMPLES)]

print(f"{REPEATS} repeats each, {SAMPLES} sentences:\n")
agreements = []
for item in sample_items:
    prompt = PROMPT.format(text=item["text"])
    labels = []
    for _ in range(REPEATS):
        try:
            labels.append(_extract_level(generate_guarded(prompt)))
        except Exception as e:
            labels.append("ERR")
    c = Counter(labels)
    modal, modal_n = c.most_common(1)[0]
    agree = modal_n / REPEATS
    agreements.append(agree)
    print(f"  gold {item['label']}  →  modal {modal} "
          f"({modal_n}/{REPEATS} = {agree:.0%})   dist={dict(c)}")

print(f"\nMean label agreement across samples: "
      f"{statistics.mean(agreements):.0%}")
print("→ Tell students their F1 can wobble by roughly the complement of this.")

## Go / no-go

- **Step 2 fails** → `colab.ai` unavailable on Tohoku accounts → use the Gemini-API
  `.env` fallback (D4).
- **Step 3 throttles per-minute** → expected and fixable. Read the observed RPM off
  Step 3, set `ASSUMED_RPM`, and confirm with Step 3b.
- **Step 3 throttles per-day** → a hard budget. Size the labs to fit it, or move to
  the Gemini API with `gemini-3.1-flash-lite` (500/day).
- **Step 3b still fails** → pacing is not enough on this account → fallback.
- **Step 3b's seconds-per-call** → multiply by the per-student call count to get the
  real lab duration. If a lab exceeds the class period, shrink the gold set.
- **Step 5 agreement low (<~80%)** → warn students about F1 wobble; consider
  majority-vote over repeats.

Record the numbers in `planning/course_planning/prep-plan.md` (milestone 07-09).

The guards used here are explained, with runnable no-key demos, in
[`resources/extra/handling-rate-limits.ipynb`](../extra/handling-rate-limits.ipynb).